# PL6 · IA Generativa
## Pré-treino e pós-treino na prática (120 min)

**Mestrado em IA · Prática Laboratorial**

Nesta sessão vai reproduzir, em ponto pequeno, as decisões centrais do pipeline de um LLM:

| Parte | Exercício | Tempo | GPU |
|---|---|---|---|
| A · Pré-treino | 1. Curadoria: filtros heurísticos, deduplicação exata e MinHash-LSH | 25 min | não |
| | 2. Filtro de qualidade com classificador + contas de escala | 15 min | opcional |
| B · Pós-treino | 3. DPO de raiz: log-probabilidades, perda e um passo de gradiente | 20 min | sim |
| | 4. DPO com TRL + LoRA e análise das métricas | 25 min | sim |
| | 5. GRPO com recompensas verificáveis (GSM8K) | 20 min | sim |

**Ambiente recomendado:** Google Colab com GPU T4 (16 GB) — *Ambiente de execução → Alterar tipo → T4 GPU*.

**Convenções**
- `# TODO` assinala código a completar; as células seguintes têm `assert` para verificar a solução.
- ✍️ assinala perguntas a responder por escrito na própria célula (contam para a avaliação).
- Os tempos de treino indicados assumem uma T4; se ficar atrasado, reduza `max_steps`.

## 0 · Preparação (≈ 5 min)

In [ ]:
# Instalação no Colab (≈ 1–2 min). Se o Colab pedir para reiniciar a sessão, reinicie e continue na célula seguinte.
!pip -q install "trl==0.19.1" "peft>=0.15" "datasets>=3.0" "datasketch>=1.6" accelerate

In [ ]:
import os, re, math, random, time, zlib, hashlib, gc
from collections import Counter, defaultdict
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, torch.nn.functional as F

SEMENTE = 0
random.seed(SEMENTE); np.random.seed(SEMENTE); torch.manual_seed(SEMENTE)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "(sem GPU: as Partes 3–5 vão ser muito lentas)")

---
# PARTE A · PRÉ-TREINO

## 1 · Curadoria de um corpus web (≈ 25 min)

Vamos usar ~3 000 documentos em português do **FineWeb-2** (`por_Latn`). Como este corpus **já foi filtrado e deduplicado**, vamos **injetar corrupção controlada** para termos verdade de base:

- 50 **duplicados exatos**;
- 100 **quase-duplicados** (palavras apagadas ou trocadas com uma dada taxa);
- 50 documentos **lixo** (menus, hashtags, repetição, *lorem ipsum*).

Assim conseguimos medir *precisão* e *recall* de cada etapa do funil.

In [ ]:
from datasets import load_dataset

N_DOCS = 3000

def corpus_sintetico(n=N_DOCS, seed=0):
    """Plano B (sem internet): pseudo-texto com vocabulário português."""
    rng = random.Random(seed)
    vocab = ("o a de que e do da em um para com não uma os no se na por mais as dos como mas foi ao ele das tem "
             "seu sua ou ser quando muito há nos já está também só pelo pela até isso ela entre era depois sem mesmo "
             "cidade governo universidade estudo dados modelo sistema projeto trabalho tempo ano país mundo forma parte "
             "vida história caso problema lugar empresa grupo região investigação resultado processo").split()
    docs = []
    for _ in range(n):
        alvo, frases, w = rng.randint(30, 600), [], 0
        while w < alvo:
            k = rng.randint(6, 20); w += k
            frases.append(" ".join(rng.choice(vocab) for _ in range(k)).capitalize() + ".")
        docs.append(" ".join(frases))
    return docs

def carregar_corpus(n=N_DOCS):
    for nome, cfg in [("HuggingFaceFW/fineweb-2", "por_Latn"), ("HuggingFaceFW/fineweb", "sample-10BT")]:
        try:
            ds = load_dataset(nome, name=cfg, split="train", streaming=True)
            docs = [x["text"] for _, x in zip(range(n), ds)]
            print(f"Carregados {len(docs)} documentos de {nome} [{cfg}]")
            return docs
        except Exception as e:
            print(f"Falhou {nome} [{cfg}]: {type(e).__name__}")
    print("A usar corpus sintético (resultados qualitativamente semelhantes, textos sem sentido).")
    return corpus_sintetico(n)

docs = carregar_corpus()
print("Palavras por documento (mediana):", int(np.median([len(d.split()) for d in docs])))

In [ ]:
def quase_duplicado(texto, taxa, rng):
    """Apaga (taxa/2) ou substitui (taxa/2) cada palavra com probabilidade `taxa`."""
    palavras, out = texto.split(), []
    for p in palavras:
        u = rng.random()
        if u < taxa / 2:   continue
        elif u < taxa:     out.append(rng.choice(palavras))
        else:              out.append(p)
    return " ".join(out)

LIXO = [
    lambda rng: "\n".join("• " + rng.choice(["Início", "Contactos", "Loja", "Login", "Carrinho", "Termos", "Privacidade"]) for _ in range(40)),
    lambda rng: " ".join(rng.choice(["#promo", "#oferta", "...", "$$$", "#grátis", "#top"]) for _ in range(200)),
    lambda rng: "clique aqui para saber mais " * 40,
    lambda rng: "Lorem ipsum dolor sit amet, consectetur adipiscing elit. " * 20,
]

def corromper(docs, n_exatos=50, n_quase=100, n_lixo=50, taxa=0.02, seed=0):
    rng = random.Random(seed)
    corpus = [{"id": i, "texto": t, "tipo": "original", "origem": None} for i, t in enumerate(docs)]
    base = [d for d in corpus if len(d["texto"].split()) >= 100]
    for _ in range(n_exatos):
        d = rng.choice(base); corpus.append({"id": len(corpus), "texto": d["texto"], "tipo": "exato", "origem": d["id"]})
    for _ in range(n_quase):
        d = rng.choice(base); corpus.append({"id": len(corpus), "texto": quase_duplicado(d["texto"], taxa, rng), "tipo": "quase", "origem": d["id"]})
    for _ in range(n_lixo):
        corpus.append({"id": len(corpus), "texto": rng.choice(LIXO)(rng), "tipo": "lixo", "origem": None})
    rng.shuffle(corpus)
    return corpus

corpus = corromper(docs)
print(pd.Series([d["tipo"] for d in corpus]).value_counts())

### 1.1 · Filtros heurísticos (estilo Gopher / C4)

Complete as regras em falta. Referência (Rae et al., 2021): 50–100 000 palavras; comprimento médio das palavras entre 3 e 10; rácio símbolos/palavras ≤ 0,1 (contar `#`, `...` e `…`); ≤ 90 % das linhas a começar por marcador; ≤ 30 % das linhas a acabar em reticências; ≥ 80 % das palavras com pelo menos uma letra; pelo menos 2 *stopwords*. E um filtro de repetição: a fração de caracteres ocupada pelo bigrama mais frequente deve ser ≤ 0,20.

In [ ]:
STOP = {"de", "a", "o", "que", "e", "do", "da", "em", "um", "para", "é", "com", "não", "uma", "os", "no", "se", "na", "por",
        "the", "be", "to", "of", "and", "that", "have", "with"}

def filtro_gopher(texto, stop=STOP):
    palavras = texto.split()
    n = len(palavras)
    if not (50 <= n <= 100_000):
        return False, "n_palavras"
    media = sum(len(p) for p in palavras) / n
    if not (3 <= media <= 10):
        return False, "comprimento_medio"
    n_simb = texto.count("#") + texto.count("...") + texto.count("…")
    if n_simb / n > 0.1:
        return False, "simbolos"
    linhas = [l.strip() for l in texto.split("\n") if l.strip()]
    if linhas:
        if sum(l.startswith(("•", "-", "*", "·", "–")) for l in linhas) / len(linhas) > 0.9:
            return False, "marcadores"
        if sum(l.endswith(("...", "…")) for l in linhas) / len(linhas) > 0.3:
            return False, "reticencias"
    if sum(bool(re.search(r"[^\W\d_]", p)) for p in palavras) / n < 0.8:
        return False, "alfabeticas"
    if sum(p.lower().strip(".,;:!?\"'()«»") in stop for p in palavras) < 2:
        return False, "stopwords"
    return True, "ok"

def frac_top_ngrama(texto, n=2):
    p = texto.lower().split()
    grams = [" ".join(p[i:i + n]) for i in range(len(p) - n + 1)]
    if not grams:
        return 0.0
    g, c = Counter(grams).most_common(1)[0]
    return c * len(g.replace(" ", "")) / max(1, sum(len(x) for x in p))

def filtro_repeticao(texto, limiar=0.20):
    f = frac_top_ngrama(texto, 2)
    return (True, "ok") if f <= limiar else (False, "repeticao")

def aplicar_filtros(texto):
    ok, motivo = filtro_gopher(texto)
    if not ok:
        return motivo
    ok, motivo = filtro_repeticao(texto)
    return "ok" if ok else motivo

In [ ]:
for d in corpus:
    d["filtro"] = aplicar_filtros(d["texto"])
tab = pd.crosstab(pd.Series([d["tipo"] for d in corpus], name="tipo"), pd.Series([d["filtro"] for d in corpus], name="resultado"))
display(tab)

lixo = [d for d in corpus if d["tipo"] == "lixo"]
assert sum(d["filtro"] != "ok" for d in lixo) == len(lixo), "Todo o lixo injetado devia ser removido pelos filtros."
print("✔ Todo o lixo foi removido.")

✍️ **P1.1** Que regras removem cada tipo de lixo? Há documentos *originais* removidos? Mostre 2 exemplos e discuta se são falsos positivos ou lixo real.

**✍️ Resposta:** 

Cada tipo de lixo é apanhado por uma regra diferente: os menus pela regra dos marcadores (todas as linhas começam por "•"), as hashtags pelo rácio de símbolos (muitos "#" e "..."), o "clique aqui para saber mais" repetido pelo filtro de repetição (passa as stopwords por causa do "para", mas o bigrama mais frequente ocupa muito mais de 20% dos caracteres) e o lorem ipsum pela regra das stopwords (não tem nenhuma palavra funcional portuguesa ou inglesa).

Há documentos originais removidos, quase todos pela regra do número mínimo de palavras (menos de 50). Os dois exemplos são textos curtos com conteúdo normal, por isso são falsos positivos: a regra foi pensada para a web em inglês e elimina textos curtos legítimos (notícias breves, descrições). Ainda assim, uma parte destes textos curtos tem pouco valor para pré-treino, o que torna a regra aceitável como compromisso.

In [ ]:
for d in [d for d in corpus if d["tipo"] == "original" and d["filtro"] != "ok"][:2]:
    print(f'[{d["filtro"]}]', d["texto"][:300].replace("\n", " ⏎ "), "\n")

### 1.2 · Deduplicação exata

Normalize o texto (minúsculas, espaços colapsados), calcule um hash e mantenha, em cada grupo, o documento com **menor `id`** (o original).

In [ ]:
def normalizar(texto):
    return " ".join(texto.lower().split())

def dedup_exata(docs_):
    grupos = defaultdict(list)
    for d in docs_:
        grupos[hashlib.md5(normalizar(d["texto"]).encode()).hexdigest()].append(d)
    manter = {min(g, key=lambda d: d["id"])["id"] for g in grupos.values()}
    return [d for d in docs_ if d["id"] in manter], [d for d in docs_ if d["id"] not in manter]

apos_filtros = [d for d in corpus if d["filtro"] == "ok"]
apos_exata, removidos = dedup_exata(apos_filtros)
print(f"{len(apos_filtros)} → {len(apos_exata)} documentos;", Counter(d["tipo"] for d in removidos))
assert Counter(d["tipo"] for d in removidos)["exato"] == 50

### 1.3 · MinHash de raiz

Para dois conjuntos de *shingles* (5-gramas de palavras) $A$ e $B$:

$$J(A,B)=\frac{|A\cap B|}{|A\cup B|},\qquad P\big[\min h(A)=\min h(B)\big]=J(A,B)$$

Implemente a assinatura com $k$ funções de hash universais $h_j(x)=(a_j x+b_j) \bmod p$ e verifique que a fração de posições iguais estima $J$.

In [ ]:
def shingles(texto, n=5):
    p = re.findall(r"\w+", texto.lower())
    return {" ".join(p[i:i + n]) for i in range(max(1, len(p) - n + 1))}

def jaccard(a, b):
    return len(a & b) / len(a | b) if (a | b) else 0.0

P_PRIMO = np.uint64(4294967311)  # primo > 2^32
def assinatura_minhash(sh, k=112, seed=1):
    rng = np.random.default_rng(seed)               # a mesma semente para todos os documentos!
    a = rng.integers(1, 2**31, size=k, dtype=np.uint64)
    b = rng.integers(0, 2**31, size=k, dtype=np.uint64)
    x = np.fromiter((zlib.crc32(s.encode()) for s in sh), dtype=np.uint64, count=len(sh))
    h = (a[:, None] * x[None, :] + b[:, None]) % P_PRIMO
    return h.min(axis=1)

def jaccard_estimado(s1, s2):
    return float(np.mean(s1 == s2))

In [ ]:
rng = random.Random(1)
base = [d["texto"] for d in apos_exata if len(d["texto"].split()) >= 150]
linhas = []
for taxa in [0.01, 0.03, 0.05, 0.10, 0.20]:
    for _ in range(5):
        a = rng.choice(base); b = quase_duplicado(a, taxa, rng)
        sa, sb = shingles(a), shingles(b)
        for k in [16, 112, 512]:
            linhas.append({"taxa": taxa, "k": k, "J_exato": jaccard(sa, sb),
                           "J_estimado": jaccard_estimado(assinatura_minhash(sa, k), assinatura_minhash(sb, k))})
res = pd.DataFrame(linhas); res["erro_abs"] = (res.J_estimado - res.J_exato).abs()
display(res.groupby("k").erro_abs.mean().rename("erro absoluto médio"))
assert res[res.k == 512].erro_abs.mean() < 0.05

✍️ **P1.2** Como varia o erro com $k$? Relacione com o desvio-padrão do estimador, $\sqrt{J(1-J)/k}$. Porque é que 5 % de palavras alteradas dão um $J$ tão baixo?

**✍️ Resposta:** 

O erro diminui com $k$: cerca de 0,09 com $k=16$, 0,04 com $k=112$ e 0,014 com $k=512$. Isto segue o desvio-padrão do estimador $\sqrt{J(1-J)/k}$, que cai com $1/\sqrt{k}$: multiplicar $k$ por 4 reduz o erro para metade. Para $J\approx0{,}6$, dá cerca de 0,12, 0,05 e 0,02, e o erro absoluto médio é um pouco menor que o desvio-padrão.

Com 5% de palavras alteradas o $J$ fica baixo porque cada shingle tem 5 palavras e só se mantém se nenhuma das 5 for alterada: $0{,}95^5\approx0{,}77$. Os shingles alterados passam a ser novos e também entram na união, por isso $J\approx0{,}77/(2-0{,}77)\approx0{,}63$, que bate com o valor medido.

### 1.4 · LSH por bandas e curva em S

Divida a assinatura em $b$ bandas de $r$ linhas: dois documentos são **candidatos** se coincidirem em **todas** as linhas de **pelo menos uma** banda. Teoricamente:

$$P(\text{candidato}\mid s)=1-(1-s^{r})^{b}$$

O FineWeb usa $b=14$, $r=8$ ($k=112$).

In [ ]:
def candidato_lsh(s1, s2, b=14, r=8):
    assert len(s1) == b * r
    return any(np.array_equal(s1[i * r:(i + 1) * r], s2[i * r:(i + 1) * r]) for i in range(b))

def p_teorico(s, b=14, r=8):
    return 1 - (1 - s ** r) ** b

rng = random.Random(2)
pontos = []
for taxa in [0.005, 0.01, 0.02, 0.03, 0.05, 0.08, 0.12]:
    Js, det = [], 0
    for _ in range(80):
        a = rng.choice(base); bq = quase_duplicado(a, taxa, rng)
        sa, sb = shingles(a), shingles(bq)
        Js.append(jaccard(sa, sb)); det += candidato_lsh(assinatura_minhash(sa), assinatura_minhash(sb))
    pontos.append((np.mean(Js), det / 80))

s = np.linspace(0, 1, 200)
plt.figure(figsize=(6, 3.5))
plt.plot(s, p_teorico(s), label="teórico b=14, r=8")
plt.scatter(*zip(*pontos), color="C1", zorder=3, label="empírico")
plt.axvline((1 / 14) ** (1 / 8), ls="--", c="gray", lw=1, label="limiar ≈ (1/b)^(1/r)")
plt.xlabel("Jaccard s"); plt.ylabel("P(candidato)"); plt.legend(); plt.grid(alpha=.3); plt.show()

✍️ **P1.3** Os pontos empíricos seguem a curva? A partir de que taxa de edição os quase-duplicados deixam de ser apanhados? Que $(b,r)$ escolheria para apanhar $s\ge 0{,}6$ com o mesmo $k=112$?

**✍️ Resposta:** 

Sim, os pontos empíricos seguem a curva teórica, com pequenas diferenças devidas ao ruído das 80 amostras. Até cerca de 2% de edição ($J\approx0{,}84$) quase todos os quase-duplicados são apanhados. Com 3% ($J\approx0{,}77$) já falham cerca de um quarto, com 5% ($J\approx0{,}64$) só cerca de um terço é detetado, e a partir de 8% quase nenhum. Isto acontece porque o limiar $(1/14)^{1/8}\approx0{,}72$ fica acima destes valores de $J$.

Para apanhar $s\ge0{,}6$ com $k=112$ escolheria $b=28$, $r=4$: o limiar desce para $\approx0{,}44$ e $P(\text{candidato}\mid0{,}6)\approx0{,}98$. O custo é haver mais falsos candidatos ($P\approx0{,}52$ para $s=0{,}4$), o que obriga a verificar mais pares. A opção $b=16$, $r=7$ não chega, porque o limiar fica em $\approx0{,}67$.

### 1.5 · Deduplicação à escala do corpus (datasketch)

In [ ]:
from datasketch import MinHash, MinHashLSH

def minhash_ds(sh, k=112):
    m = MinHash(num_perm=k, seed=1)
    m.update_batch([s.encode("utf8") for s in sh])
    return m

def dedup_quase(docs_, b=14, r=8):
    lsh, mhs = MinHashLSH(num_perm=b * r, params=(b, r)), {}
    for d in docs_:
        mhs[d["id"]] = minhash_ds(shingles(d["texto"]), b * r); lsh.insert(d["id"], mhs[d["id"]])
    pai = {d["id"]: d["id"] for d in docs_}
    def raiz(x):
        while pai[x] != x:
            pai[x] = pai[pai[x]]; x = pai[x]
        return x
    for d in docs_:
        for c in lsh.query(mhs[d["id"]]):
            ra, rb = raiz(d["id"]), raiz(c)
            if ra != rb: pai[max(ra, rb)] = min(ra, rb)   # o representante é o menor id
    grupo = {d["id"]: raiz(d["id"]) for d in docs_}
    return [d for d in docs_ if grupo[d["id"]] == d["id"]], grupo

t0 = time.time()
final, grupo = dedup_quase(apos_exata)
quase = [d for d in apos_exata if d["tipo"] == "quase"]
recall = np.mean([grupo[d["id"]] != d["id"] for d in quase])
orig_rem = [d for d in apos_exata if d["tipo"] == "original" and grupo[d["id"]] != d["id"]]
print(f"{len(apos_exata)} → {len(final)} documentos em {time.time() - t0:.0f}s")
print(f"Recall nos quase-duplicados injetados: {recall:.1%}  |  originais removidos: {len(orig_rem)}")

In [ ]:
# Funil completo
funil = pd.Series({"bruto": len(corpus), "após filtros": len(apos_filtros), "após dedup. exata": len(apos_exata), "após MinHash-LSH": len(final)})
funil.plot.barh(figsize=(6, 2.5), color="C1"); plt.gca().invert_yaxis(); plt.title("Funil de curadoria"); plt.show()
for d in orig_rem[:2]:
    rep = next(x for x in apos_exata if x["id"] == grupo[d["id"]])
    print(f'J = {jaccard(shingles(d["texto"]), shingles(rep["texto"])):.2f}\n  A: {d["texto"][:160]!r}\n  B: {rep["texto"][:160]!r}\n')

✍️ **P1.4** Os originais removidos pelo MinHash são erros ou duplicados *naturais* do corpus? O que mudaria se a deduplicação fosse global sobre vários *snapshots* do Common Crawl (recorde o resultado do FineWeb)?

**✍️ Resposta:** 

Os originais removidos pelo MinHash não são erros do algoritmo: são duplicados naturais que já existiam no corpus, como páginas geradas pelo mesmo modelo, avisos legais, a mesma notícia publicada em vários sites ou textos com o mesmo boilerplate. O $J$ entre eles e o representante é alto, por isso fazem sentido como quase-duplicados.

Se a deduplicação fosse global sobre vários snapshots do Common Crawl, seria removida muito mais informação, porque o mesmo conteúdo aparece em crawls diferentes. O FineWeb mostrou que isso não compensa: a deduplicação global retirava a maior parte dos dados dos snapshots antigos e o que sobrava era de pior qualidade, e os modelos treinados assim não melhoravam. Deduplicar cada snapshot separadamente deu melhores resultados.

## 2 · Filtro de qualidade e contas de escala (≈ 15 min)

### 2.1 · Classificador educativo do FineWeb-Edu

O `HuggingFaceFW/fineweb-edu-classifier` é um regressor (embeddings Snowflake-Arctic + cabeça linear) treinado com ~450 mil notas 0–5 atribuídas pelo Llama-3-70B. O FineWeb-Edu mantém os documentos com nota arredondada ≥ 3.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
CLS = "HuggingFaceFW/fineweb-edu-classifier"
tok_c = AutoTokenizer.from_pretrained(CLS)
clf = AutoModelForSequenceClassification.from_pretrained(CLS).to(DEVICE).eval()

@torch.no_grad()
def nota_edu(textos, bs=16):
    notas = []
    for i in range(0, len(textos), bs):
        enc = tok_c(textos[i:i + bs], return_tensors="pt", padding="longest", truncation=True, max_length=512).to(DEVICE)
        notas += clf(**enc).logits.squeeze(-1).float().cpu().tolist()
    return np.array(notas)

en = load_dataset("HuggingFaceFW/fineweb", name="sample-10BT", split="train", streaming=True)
docs_en = [x["text"] for _, x in zip(range(300), en)]
notas_en = nota_edu(docs_en)
notas_pt = nota_edu([d["texto"] for d in final[:300]])

plt.figure(figsize=(6, 3)); plt.hist(notas_en, bins=30, alpha=.6, label="FineWeb (EN)"); plt.hist(notas_pt, bins=30, alpha=.6, label="FineWeb-2 (PT)")
plt.axvline(2.5, c="k", ls="--", lw=1); plt.legend(); plt.xlabel("nota educativa"); plt.show()
print(f"Fração com nota ≥ 3 — EN: {(notas_en >= 2.5).mean():.1%} | PT: {(notas_pt >= 2.5).mean():.1%}")
for i in list(np.argsort(notas_en)[-2:]) + list(np.argsort(notas_en)[:2]):
    print(f"\n[{notas_en[i]:.2f}]", docs_en[i][:250].replace("\n", " "))

✍️ **P2.1** Que fração passa o limiar? Que características têm os documentos com nota alta e baixa? O que acontece com os documentos em português e que viés isso introduziria num corpus multilingue?

**✍️ Resposta:** 

Só uma pequena fração dos documentos em inglês passa o limiar de 3, o que está de acordo com o FineWeb-Edu manter apenas uma pequena parte do FineWeb. Os documentos com nota alta são textos explicativos e educativos (ciência, história, tutoriais, conteúdo tipo manual). Os de nota baixa são páginas comerciais, publicidade, notícias de entretenimento, fóruns e texto pouco informativo.

Os documentos em português recebem notas muito mais baixas e quase nenhum passa o limiar, porque o classificador foi treinado com embeddings e anotações em inglês e penaliza a língua e não só o conteúdo. Num corpus multilingue isto criava um viés forte para o inglês: as outras línguas seriam quase todas eliminadas e ficariam sub-representadas. Seria preciso um classificador por língua ou limiares ajustados para cada língua.

### 2.2 · Calculadora de pré-treino

Implemente as funções e responda às perguntas. Use o pico denso BF16 da H100 (989 TFLOP/s).

In [ ]:
def flops_treino(N, D):
    return 6 * N * D

def dias_treino(N, D, n_gpus, pico=989e12, mfu=0.40):
    return flops_treino(N, D) / (n_gpus * pico * mfu) / 86400

def memoria_estados_gb(psi, nd=1, estagio=0, K=12):
    """Pesos+gradientes BF16 (2+2 bytes) + estados Adam (K=12 bytes), com ZeRO-0/1/2/3."""
    if estagio == 0:
        b = 2 + 2 + K
    elif estagio == 1:
        b = 2 + 2 + K / nd
    elif estagio == 2:
        b = 2 + (2 + K) / nd
    else:
        b = (2 + 2 + K) / nd
    return b * psi / 1e9

assert abs(flops_treino(405e9, 15.6e12) / 3.8e25 - 1) < 0.02
assert abs(dias_treino(70e9, 15e12, 2048) - 90) < 2
assert memoria_estados_gb(70e9, 64, 3) == 17.5
print("✔ Calculadora correta")

In [ ]:
# Q-a) Quantas H100 para treinar um 8B com 15T tokens em 30 dias com MFU de 40%?
n_gpus = math.ceil(flops_treino(8e9, 15e12) / (30 * 86400 * 989e12 * 0.40))
print("GPUs necessárias:", n_gpus)

# Q-b) Afinação completa com Adam numa T4 (16 GB): cabe um 0,5B? E um 7B?
for psi in [0.5e9, 7e9]:
    print(f"{psi/1e9:.1f}B → {memoria_estados_gb(psi):.0f} GB só em estados (sem ativações)")

✍️ **P2.2** Interprete os resultados de (a) e (b). Porque é que, nas Partes 4–5, usamos LoRA mesmo num modelo de 0,5B?

**✍️ Resposta:** 

(a) Treinar um modelo de 8B com 15T tokens em 30 dias exige cerca de 703 H100 a 40% de MFU, ou seja, mais de 20 mil GPU-dias, o que mostra o custo enorme do pré-treino mesmo para um modelo pequeno. (b) Na afinação completa com Adam, um modelo de 0,5B já precisa de 8 GB só para pesos, gradientes e estados do otimizador, sem contar ativações, por isso fica no limite de uma T4 de 16 GB. Um 7B precisaria de 112 GB, o que é impossível.

Usamos LoRA mesmo no modelo de 0,5B porque os pesos originais ficam congelados e só se guardam gradientes e estados do Adam para os adaptadores (cerca de 1–2% dos parâmetros). Isto liberta memória para as ativações, para a referência do DPO (que é o mesmo modelo com os adaptadores desligados) e para as gerações do GRPO, e torna o treino mais rápido.

---
# PARTE B · PÓS-TREINO

## 3 · DPO de raiz (≈ 20 min)

$$\mathcal{L}_{\text{DPO}}=-\log\sigma\!\Big(\beta\log\frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)}-\beta\log\frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)}\Big)$$

Usamos `Qwen/Qwen2.5-0.5B-Instruct` como política e como referência.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
MODELO = "Qwen/Qwen2.5-0.5B-Instruct"
BF16 = DEVICE == "cuda" and torch.cuda.is_bf16_supported()
tok = AutoTokenizer.from_pretrained(MODELO)
politica = AutoModelForCausalLM.from_pretrained(MODELO, torch_dtype=torch.float32).to(DEVICE)
referencia = AutoModelForCausalLM.from_pretrained(MODELO, torch_dtype=torch.float32).to(DEVICE).eval()
for p in referencia.parameters(): p.requires_grad_(False)
print(f"{sum(p.numel() for p in politica.parameters())/1e6:.0f}M parâmetros")

### 3.1 · Log-probabilidade de uma resposta

$\log\pi(y|x)=\sum_t \log\pi(y_t\mid x,y_{<t})$ — **só nos tokens da resposta** (incluindo o fim de turno). Atenção ao desvio de uma posição entre *logits* e alvos.

In [ ]:
def logprob_resposta(modelo, prompt, resposta, grad=False):
    txt_p = tok.apply_chat_template([{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True)
    ids_p = tok(txt_p, add_special_tokens=False, return_tensors="pt").input_ids
    ids_r = tok(resposta + tok.eos_token, add_special_tokens=False, return_tensors="pt").input_ids
    ids = torch.cat([ids_p, ids_r], dim=1).to(modelo.device)
    with torch.set_grad_enabled(grad):
        logits = modelo(ids).logits[:, :-1].float()          # logits[t] prevê ids[t+1]
        alvo = ids[:, 1:]
        lp = torch.log_softmax(logits, -1).gather(-1, alvo.unsqueeze(-1)).squeeze(-1)
        mascara = torch.zeros_like(alvo, dtype=torch.bool)
        mascara[:, ids_p.shape[1] - 1:] = True
        return lp[mascara].sum()

### 3.2 · A perda

In [ ]:
def perda_dpo(lp_pol_w, lp_pol_l, lp_ref_w, lp_ref_l, beta=0.1):
    """Devolve (perda, recompensa implícita de y_w, recompensa implícita de y_l)."""
    rw = beta * (lp_pol_w - lp_ref_w)
    rl = beta * (lp_pol_l - lp_ref_l)
    perda = -F.logsigmoid(rw - rl)
    return perda, rw, rl

# Verificação com o Exercício II da aula teórica: diferenças +2 e −1, β = 0,1  →  perda ≈ 0,554
L, rw, rl = perda_dpo(torch.tensor(2.), torch.tensor(-1.), torch.tensor(0.), torch.tensor(0.))
assert abs(L.item() - 0.5544) < 1e-3
print(f"perda = {L.item():.4f}   peso do gradiente σ(r_l − r_w) = {torch.sigmoid(rl - rw).item():.3f}")

In [ ]:
PROMPT = "Explica em duas frases o que é a deduplicação de dados no pré-treino de LLMs."
Y_W = ("É a remoção de documentos repetidos ou quase repetidos do corpus de treino. Evita que o modelo memorize "
       "texto duplicado e poupa compute, além de reduzir a contaminação de benchmarks.")
Y_L = "Deduplicação é quando os dados são duplicados para o modelo aprender melhor."

lp = {nome: logprob_resposta(m, PROMPT, y).item() for nome, m, y in
      [("pol_w", politica, Y_W), ("pol_l", politica, Y_L), ("ref_w", referencia, Y_W), ("ref_l", referencia, Y_L)]}
L0, _, _ = perda_dpo(torch.tensor(lp["pol_w"]), torch.tensor(lp["pol_l"]), torch.tensor(lp["ref_w"]), torch.tensor(lp["ref_l"]))
print(lp); print(f"perda inicial = {L0.item():.4f}   (log 2 = {math.log(2):.4f})")

✍️ **P3.1** Porque é que a perda inicial é exatamente $\log 2$? Qual das duas respostas tem maior $\log\pi$ e porque é que isso **não** significa que o modelo a prefira?

**✍️ Resposta:** 

No início a política e a referência são o mesmo modelo, por isso $\log\pi_\theta-\log\pi_{\text{ref}}=0$ para as duas respostas. As duas recompensas implícitas são 0 e a perda é $-\log\sigma(0)=\log2$.

A resposta rejeitada tem maior $\log\pi$ porque é mais curta: o $\log\pi$ é uma soma de termos negativos, um por token, e uma resposta com menos tokens tem menos termos a somar. Isto não quer dizer que o modelo a prefira, porque o valor absoluto depende sobretudo do comprimento e da frequência dos tokens, não da qualidade. O DPO só compara cada resposta com a referência, e no início as duas razões são iguais.

### 3.3 · Dez passos de gradiente num único par

Implemente o ciclo de treino (SGD, `lr=1e-5`) e registe $\log\pi_\theta(y_w)$, $\log\pi_\theta(y_l)$ e a margem.

In [ ]:
otim = torch.optim.SGD(politica.parameters(), lr=1e-5)
hist = []
for passo in range(10):
    lw = logprob_resposta(politica, PROMPT, Y_W, grad=True)
    ll = logprob_resposta(politica, PROMPT, Y_L, grad=True)
    perda, rw, rl = perda_dpo(lw, ll, torch.tensor(lp["ref_w"], device=lw.device), torch.tensor(lp["ref_l"], device=lw.device))
    otim.zero_grad()
    perda.backward()
    otim.step()
    hist.append({"passo": passo, "perda": perda.item(), "logp_w": lw.item(), "logp_l": ll.item(), "margem": (rw - rl).item()})
h = pd.DataFrame(hist); display(h.round(3))
h.plot(x="passo", y=["logp_w", "logp_l"], figsize=(6, 3), marker="o"); plt.grid(alpha=.3); plt.show()

✍️ **P3.2** A margem cresce? O que acontece a $\log\pi_\theta(y_w)$ — sobe, desce ou fica? Relacione com o *deslocamento de verosimilhança* (Razin et al., 2024). Que papel tem $\beta$?

**✍️ Resposta:** 

A margem cresce a cada passo e a perda desce abaixo de $\log2$. O $\log\pi_\theta(y_w)$ não sobe como seria de esperar e tende a descer, embora menos do que o $\log\pi_\theta(y_l)$. É o deslocamento de verosimilhança (Razin et al., 2024): o DPO só garante que a diferença entre as duas aumenta. Como as respostas partilham muitos tokens, empurrar $y_l$ para baixo também puxa $y_w$ para baixo, e a probabilidade vai para respostas que não estão no par.

O $\beta$ controla a força da ligação à referência: escala as recompensas implícitas e o peso do gradiente. Com $\beta$ maior, afastar-se da referência é mais penalizado e as mudanças são mais contidas. Com $\beta$ menor, a política pode afastar-se mais.

In [ ]:
del politica, referencia, otim; gc.collect(); torch.cuda.empty_cache()   # libertar a GPU

## 4 · DPO com TRL + LoRA (≈ 25 min, ~6 min de treino)

Dados: 1 000 pares do `trl-lib/ultrafeedback_binarized`. A referência é o próprio modelo com os adaptadores LoRA desligados.

In [ ]:
from datasets import load_dataset
from trl import DPOConfig, DPOTrainer
from peft import LoraConfig

pref = load_dataset("trl-lib/ultrafeedback_binarized", split="train").shuffle(seed=0).select(range(1000))
print(pref[0]["chosen"][-1]["content"][:200])

lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules="all-linear", task_type="CAUSAL_LM")
modelo = AutoModelForCausalLM.from_pretrained(MODELO, torch_dtype=torch.bfloat16 if BF16 else torch.float32)

cfg_dpo = DPOConfig(
    output_dir="dpo-qwen05", beta=0.1, learning_rate=5e-5, lr_scheduler_type="cosine", warmup_ratio=0.1,
    per_device_train_batch_size=2, gradient_accumulation_steps=4, max_steps=40,
    max_length=384, max_prompt_length=192, gradient_checkpointing=True,
    bf16=BF16, fp16=(DEVICE == "cuda" and not BF16),   # T4: fp16 misto. Se a perda der NaN, ponha fp16=False.
    logging_steps=5, save_strategy="no", report_to="none",
)
treinador = DPOTrainer(model=modelo, args=cfg_dpo, train_dataset=pref, processing_class=tok, peft_config=lora)
treinador.model.print_trainable_parameters()
treinador.train()

In [ ]:
log = pd.DataFrame([h for h in treinador.state.log_history if "loss" in h and "train_runtime" not in h])
cols = [c for c in ["loss", "rewards/margins", "rewards/accuracies", "rewards/chosen", "rewards/rejected", "logps/chosen", "logps/rejected"] if c in log]
log.plot(x="step", y=cols, subplots=True, layout=(-1, 2), figsize=(10, 2.2 * math.ceil(len(cols) / 2)), legend=True); plt.tight_layout(); plt.show()

✍️ **P4.1** Descreva a evolução de `rewards/accuracies` e `rewards/margins`. As recompensas implícitas de *chosen* sobem ou descem? E `logps/chosen`? Interprete.

**✍️ Resposta:** 

A `rewards/accuracies` começa perto de 0,5, porque no início a política é igual à referência, e sobe ao longo do treino com algum ruído. A `rewards/margins` começa em 0 e cresce, o que mostra que a diferença entre as recompensas implícitas de chosen e rejected aumenta.

As recompensas implícitas de chosen tendem a descer, mas menos do que as de rejected, e o `logps/chosen` também desce. É o mesmo deslocamento de verosimilhança da Parte 3: o modelo aprende a separar as duas respostas, mas fá-lo sobretudo baixando a rejeitada e não subindo a preferida. Com 40 passos e lotes de 8 pares, as curvas têm bastante ruído.

In [ ]:
@torch.no_grad()
def gerar(modelo, prompt, max_new_tokens=150):
    ids = tok.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True, return_tensors="pt").to(modelo.device)
    out = modelo.generate(ids, max_new_tokens=max_new_tokens, do_sample=False)
    return tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True)

m = treinador.model.eval()
for pr in ["Give me three tips to write a good scientific abstract.", "Is it safe to eat raw chicken? Answer briefly."]:
    with m.disable_adapter():
        antes = gerar(m, pr)
    depois = gerar(m, pr)
    print(f"### {pr}\n--- ANTES ({len(antes.split())} palavras)\n{antes}\n--- DEPOIS ({len(depois.split())} palavras)\n{depois}\n")

✍️ **P4.2** Que diferenças observa (estilo, estrutura, comprimento)? Com 40 passos, o que é plausível ter mudado e o que não? Como verificaria se há viés de comprimento?

**✍️ Resposta:** 

Depois do DPO as respostas tendem a ser mais longas e mais estruturadas (listas, introdução e conclusão) e com um tom mais simpático e cauteloso. Com 40 passos e LoRA, é plausível que tenham mudado o estilo, o tom e o comprimento, mas não o conhecimento nem a capacidade de raciocínio, que vêm do pré-treino.

Para verificar o viés de comprimento, compararia o número de palavras antes e depois do DPO num conjunto maior de pedidos. Veria também nos dados de treino em que fração dos pares a resposta preferida é mais longa, e se a recompensa implícita está correlacionada com o comprimento. Numa avaliação com juiz, usaria uma taxa de vitória controlada pelo comprimento.

In [ ]:
del treinador, modelo, m; gc.collect(); torch.cuda.empty_cache()

## 5 · GRPO com recompensas verificáveis (≈ 20 min, ~8 min de treino)

### 5.1 · Vantagens relativas ao grupo

$$\hat A_i=\frac{r_i-\operatorname{mean}(r)}{\operatorname{std}(r)+\varepsilon}$$

In [ ]:
def vantagens_grupo(r, normalizar_std=True, eps=1e-4):
    r = np.asarray(r, dtype=float)
    a = r - r.mean()
    if normalizar_std:
        a = a / (r.std() + eps)
    return a

print(vantagens_grupo([1, 0, 1, 0]).round(3), vantagens_grupo([1, 1, 1, 1]), vantagens_grupo([1, 0, 0, 0]).round(3))
assert np.allclose(vantagens_grupo([1, 0, 1, 0]), [1, -1, 1, -1], atol=1e-3)
assert np.allclose(vantagens_grupo([1, 1, 1, 1]), 0)

✍️ **P5.1** Compare $[1,0,0,0]$ com $[1,1,1,0]$: que resposta recebe a maior vantagem absoluta em cada caso? Porque é que a divisão pelo desvio-padrão cria um viés de dificuldade (Dr. GRPO)?

**✍️ Resposta:** 

Em $[1,0,0,0]$ a única resposta certa recebe a maior vantagem ($\approx+1{,}73$, as outras $\approx-0{,}58$). Em $[1,1,1,0]$ é a única errada que recebe a maior vantagem em valor absoluto ($\approx-1{,}73$, as outras $\approx+0{,}58$). A resposta com o resultado mais raro dentro do grupo é sempre a que pesa mais.

Dividir pelo desvio-padrão cria um viés de dificuldade porque os grupos em que quase todas as respostas têm o mesmo resultado (perguntas muito fáceis ou muito difíceis) têm desvio-padrão pequeno, e as vantagens ficam ampliadas. Essas perguntas passam a pesar mais no gradiente do que as de dificuldade média. O Dr. GRPO corrige isto deixando de dividir pelo desvio-padrão.

### 5.2 · Funções de recompensa

O modelo deve terminar com `Resposta: <número>`. Implemente uma recompensa de **correção** (1,0 ou 0,0) e uma de **formato** (0,2). Na TRL, cada função recebe `completions` (lista de mensagens) e as colunas do dataset como argumentos nomeados.

In [ ]:
SISTEMA = "Resolve o problema passo a passo e termina com uma linha no formato 'Resposta: <número>'."
RE_RESP = re.compile(r"Resposta:\s*\$?\s*(-?[\d.,]+)", re.IGNORECASE)

def _texto(c):
    return c[0]["content"] if isinstance(c, list) else c

def extrair_numero(texto):
    m = RE_RESP.findall(texto)
    if not m:
        return None
    try:
        return float(m[-1].replace(",", "").rstrip("."))
    except ValueError:
        return None

def recompensa_correcao(completions, resposta, **kwargs):
    out = []
    for c, ref in zip(completions, resposta):
        n = extrair_numero(_texto(c))
        try:
            alvo = float(str(ref).replace(",", ""))
        except ValueError:
            alvo = None
        out.append(1.0 if n is not None and alvo is not None and abs(n - alvo) < 1e-6 else 0.0)
    return out

def recompensa_formato(completions, **kwargs):
    return [0.2 if RE_RESP.search(_texto(c)) else 0.0 for c in completions]

testes = [[{"role": "assistant", "content": "3 + 4 = 7.\nResposta: 7"}],
          [{"role": "assistant", "content": "Acho que é 8.\nResposta: 8."}],
          [{"role": "assistant", "content": "São 1,000 euros. Resposta: $1,000"}],
          [{"role": "assistant", "content": "A resposta é 7."}]]
print(recompensa_correcao(testes, ["7", "7", "1000", "7"]), recompensa_formato(testes))
assert recompensa_correcao(testes, ["7", "7", "1000", "7"]) == [1.0, 0.0, 1.0, 0.0]

✍️ **P5.2** Encontre pelo menos **duas** formas de a política obter recompensa sem resolver o problema, ou de ser penalizada injustamente (ex.: decimais com vírgula em português). Como corrigiria?

**✍️ Resposta:** 

1. Recompensa sem resolver: escrever só "Resposta: 0" sem raciocínio dá sempre 0,2 de formato. A política pode aprender o formato sem aprender a resolver. Corrigiria dando a recompensa de formato só quando a linha final aparece uma vez no fim, depois de algum raciocínio, ou tornando-a muito pequena face à de correção.
2. Penalização injusta com números em português: "Resposta: 2,5" passa a 25 depois de tirar a vírgula, e "Resposta: 1.000" (mil) é lido como 1,0. Uma resposta certa recebe 0. Corrigiria com uma leitura de números que trate a vírgula decimal e o ponto de milhares (por exemplo, vírgula seguida de 1–2 dígitos é decimal) e com uma tolerância na comparação.
3. Penalização por truncagem: se o raciocínio passar dos 256 tokens, a linha "Resposta:" nunca é escrita e uma resolução certa recebe 0. Corrigiria aumentando o `max_completion_length` ou tratando as respostas truncadas à parte.

In [ ]:
from trl import GRPOConfig, GRPOTrainer

def preparar(ex):
    return {"prompt": [{"role": "system", "content": SISTEMA}, {"role": "user", "content": ex["question"]}],
            "resposta": ex["answer"].split("####")[-1].strip().replace(",", "")}

gsm = load_dataset("openai/gsm8k", "main")
treino = gsm["train"].shuffle(seed=0).select(range(400)).map(preparar, remove_columns=["question", "answer"])
teste = gsm["test"].shuffle(seed=0).select(range(30)).map(preparar, remove_columns=["question", "answer"])

@torch.no_grad()
def exatidao(modelo, dados, max_new_tokens=256):
    modelo.eval(); acertos, formato = 0, 0
    for ex in dados:
        ids = tok.apply_chat_template(ex["prompt"], add_generation_prompt=True, return_tensors="pt").to(modelo.device)
        out = tok.decode(modelo.generate(ids, max_new_tokens=max_new_tokens, do_sample=False)[0, ids.shape[1]:], skip_special_tokens=True)
        acertos += recompensa_correcao([out], [ex["resposta"]])[0]; formato += recompensa_formato([out])[0] > 0
    return acertos / len(dados), formato / len(dados)

modelo = AutoModelForCausalLM.from_pretrained(MODELO, torch_dtype=torch.bfloat16 if BF16 else torch.float32).to(DEVICE)
acc0 = exatidao(modelo, teste); print(f"ANTES — exatidão: {acc0[0]:.0%} | formato: {acc0[1]:.0%}")

In [ ]:
cfg_grpo = GRPOConfig(
    output_dir="grpo-qwen05", learning_rate=2e-5, lr_scheduler_type="constant",
    per_device_train_batch_size=8, gradient_accumulation_steps=1, num_generations=4,   # 2 prompts × 4 respostas por passo
    max_prompt_length=256, max_completion_length=256, temperature=1.0,
    beta=0.04, loss_type="grpo", max_steps=40, gradient_checkpointing=True,
    bf16=BF16, fp16=(DEVICE == "cuda" and not BF16),
    logging_steps=5, save_strategy="no", report_to="none",
)
treinador = GRPOTrainer(model=modelo, reward_funcs=[recompensa_correcao, recompensa_formato],
                        args=cfg_grpo, train_dataset=treino, processing_class=tok, peft_config=lora)
treinador.train()

In [ ]:
log = pd.DataFrame([h for h in treinador.state.log_history if "train_runtime" not in h])
cols = [c for c in log.columns if any(k in c for k in ["reward", "kl", "mean_length", "zero_std", "entropy"]) and log[c].notna().any()]
log.plot(x="step", y=cols, subplots=True, layout=(-1, 2), figsize=(10, 2.2 * math.ceil(len(cols) / 2))); plt.tight_layout(); plt.show()
acc1 = exatidao(treinador.model, teste)
print(f"ANTES  — exatidão: {acc0[0]:.0%} | formato: {acc0[1]:.0%}\nDEPOIS — exatidão: {acc1[0]:.0%} | formato: {acc1[1]:.0%}")

✍️ **P5.3** Interprete as curvas: recompensa média, `reward_std`, fração de grupos com desvio nulo, KL e comprimento. A melhoria (se houver) vem da correção ou do formato? Com 30 exemplos de teste, quão confiável é a diferença?

**✍️ Resposta:** 

A recompensa média sobe, sobretudo no início, porque o modelo aprende rapidamente a terminar com "Resposta: <número>" e ganha os 0,2 de formato. O `reward_std` mostra quanta variação há dentro dos grupos. A fração de grupos com desvio nulo é alta: são grupos em que as 4 respostas estão todas certas ou todas erradas, que não dão sinal de aprendizagem. O KL começa em 0 e cresce devagar, porque o $\beta=0{,}04$ mantém a política perto da referência, e o comprimento das respostas vai-se ajustando.

A melhoria vem sobretudo do formato e muito menos da correção. Com apenas 30 exemplos de teste cada pergunta vale 3,3 pontos percentuais e o erro-padrão da exatidão é de cerca de 8–9 pontos, por isso uma diferença de poucos acertos não é estatisticamente fiável. Seriam precisos centenas de problemas de teste para tirar conclusões.

## 6 · Desafios extra (para quem terminar cedo ou como trabalho de casa)

1. **Dr. GRPO vs GRPO:** repita a Parte 5 com `loss_type="dr_grpo"` e `scale_rewards=False`. Compare o comprimento das respostas.
2. **β = 0:** treine o GRPO sem KL. O que acontece à entropia e ao KL implícito?
3. **IPO:** na Parte 4, use `loss_type="ipo"` no `DPOConfig` e compare `logps/chosen`.
4. **Português:** adapte os filtros da Parte 1 a português (stopwords, comprimento médio) e meça quantos originais deixam de ser removidos.
5. **Hacking:** suba a recompensa de formato para 1,0 e observe se a correção piora.

## 📦 Entrega

Submeta este notebook **executado**, com todas as células ✍️ respondidas, até ao fim da semana. Critérios: código correto (40 %), qualidade das respostas e interpretação dos gráficos (45 %), desafio extra (15 %).

## 6.4 · Desafio extra 4: filtros adaptados a português

Comparamos três versões sobre os documentos originais: (1) os filtros da Parte 1; (2) uma lista de *stopwords* portuguesa completa; (3) a lista portuguesa com limites ajustados ao português (mínimo de 30 palavras e comprimento médio até 12 caracteres).

In [ ]:
STOP_PT = {"de", "a", "o", "que", "e", "do", "da", "em", "um", "para", "é", "com", "não", "uma", "os", "no", "se", "na",
           "por", "mais", "as", "dos", "como", "mas", "foi", "ao", "ele", "das", "tem", "à", "seu", "sua", "ou", "ser",
           "quando", "muito", "há", "nos", "já", "está", "eu", "também", "só", "pelo", "pela", "até", "isso", "ela",
           "entre", "era", "depois", "sem", "mesmo", "aos", "ter", "seus", "quem", "nas", "me", "esse", "eles", "estão",
           "você", "tinha", "foram", "essa", "num", "nem", "suas", "meu", "às", "minha", "têm", "numa", "pelos", "elas",
           "havia", "seja", "qual", "será", "nós", "lhe", "deles", "essas", "esses", "pelas", "este", "fosse", "dele",
           "são", "sobre", "ainda", "pode", "onde", "esta", "isto", "aqui", "assim", "porque", "todos", "cada"}

def filtro_gopher_param(texto, stop, min_pal=50, med_min=3, med_max=10):
    palavras = texto.split()
    n = len(palavras)
    if not (min_pal <= n <= 100_000):
        return False, "n_palavras"
    media = sum(len(p) for p in palavras) / n
    if not (med_min <= media <= med_max):
        return False, "comprimento_medio"
    n_simb = texto.count("#") + texto.count("...") + texto.count("…")
    if n_simb / n > 0.1:
        return False, "simbolos"
    linhas = [l.strip() for l in texto.split("\n") if l.strip()]
    if linhas:
        if sum(l.startswith(("•", "-", "*", "·", "–")) for l in linhas) / len(linhas) > 0.9:
            return False, "marcadores"
        if sum(l.endswith(("...", "…")) for l in linhas) / len(linhas) > 0.3:
            return False, "reticencias"
    if sum(bool(re.search(r"[^\W\d_]", p)) for p in palavras) / n < 0.8:
        return False, "alfabeticas"
    if sum(p.lower().strip(".,;:!?\"'()«»") in stop for p in palavras) < 2:
        return False, "stopwords"
    return True, "ok"

def aplicar(texto, **kw):
    ok, motivo = filtro_gopher_param(texto, **kw)
    if not ok:
        return motivo
    ok, motivo = filtro_repeticao(texto)
    return "ok" if ok else motivo

variantes = {
    "1 · original": dict(stop=STOP),
    "2 · stopwords PT": dict(stop=STOP_PT),
    "3 · stopwords PT + limites PT": dict(stop=STOP_PT, min_pal=30, med_max=12),
}
originais = [d for d in corpus if d["tipo"] == "original"]
lixo_inj = [d for d in corpus if d["tipo"] == "lixo"]
linhas_res = []
for nome, kw in variantes.items():
    motivos = Counter(aplicar(d["texto"], **kw) for d in originais)
    linhas_res.append({"variante": nome,
                       "originais removidos": sum(v for k, v in motivos.items() if k != "ok"),
                       "lixo removido": sum(aplicar(d["texto"], **kw) != "ok" for d in lixo_inj),
                       **{k: v for k, v in motivos.items() if k != "ok"}})
display(pd.DataFrame(linhas_res).fillna(0).set_index("variante"))

**✍️ Resposta:**

Trocar a lista de stopwords pela portuguesa quase não muda o número de originais removidos, porque a regra só exige duas stopwords e a lista original já tinha as palavras funcionais portuguesas mais comuns. O que remove mais originais é o mínimo de 50 palavras: ao baixá-lo para 30 e ao alargar o comprimento médio para 12 caracteres, recupera-se uma boa parte dos textos curtos legítimos. O lixo injetado continua a ser todo removido, porque é apanhado pelas regras de marcadores, símbolos, repetição e stopwords, que não dependem destes limites.